In [54]:
import sys
from dotenv import load_dotenv
import os

# Load environment variables from the .env file
load_dotenv()

WORKSPACE_PATH = os.getenv("WORKSPACE_PATH")

# Add the parent directory to the system path
sys.path.append(str(WORKSPACE_PATH))


In [55]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import product
from src.config import RESULTS_DIR, PLOTS_DIR
from utils.dataframe_utils import read_excel_file
from utils.matplotlib_utils import save_fig_matplotlib

baseline_dir = RESULTS_DIR / "analysis_data" / "baseline"
regular_dir = RESULTS_DIR / "analysis_data" / "regular"
hist_plots_dir = PLOTS_DIR / "hist"

In [56]:
def get_xlsx_files(directory):
    return list(directory.glob("*.xlsx"))

In [57]:
baseline_files = get_xlsx_files(baseline_dir)
regular_files = get_xlsx_files(regular_dir)

In [58]:
print(baseline_files)
print(regular_files)

[WindowsPath('C:/Users/huber/OneDrive/Dokumenty/GitHub/swps_synchronization_study/results/analysis_data/baseline/hr_baseline.xlsx'), WindowsPath('C:/Users/huber/OneDrive/Dokumenty/GitHub/swps_synchronization_study/results/analysis_data/baseline/nn_baseline.xlsx'), WindowsPath('C:/Users/huber/OneDrive/Dokumenty/GitHub/swps_synchronization_study/results/analysis_data/baseline/rmssd_baseline.xlsx'), WindowsPath('C:/Users/huber/OneDrive/Dokumenty/GitHub/swps_synchronization_study/results/analysis_data/baseline/sdnn_baseline.xlsx')]
[WindowsPath('C:/Users/huber/OneDrive/Dokumenty/GitHub/swps_synchronization_study/results/analysis_data/regular/hr_results.xlsx'), WindowsPath('C:/Users/huber/OneDrive/Dokumenty/GitHub/swps_synchronization_study/results/analysis_data/regular/nn_results.xlsx'), WindowsPath('C:/Users/huber/OneDrive/Dokumenty/GitHub/swps_synchronization_study/results/analysis_data/regular/rmssd_results.xlsx'), WindowsPath('C:/Users/huber/OneDrive/Dokumenty/GitHub/swps_synchronizati

In [59]:
def generate_histograms(col, file_paths, plots_dir):
    """
    Generates percentage-based histograms for a given column across multiple Excel files.

    :param col: Column name for which histograms are to be generated.
    :param file_paths: List of file paths to Excel files.
    :param plots_dir: Directory where plots should be saved.
    :raises ValueError: If required columns are missing in the data.
    """

    required_columns = {"meas_number", "condition", "task", col}

    for file_path in file_paths:
        # Read the Excel file
        df = pd.read_excel(file_path)
        df.columns = (
            df.columns.str.strip().str.lower()
        )  # Clean and standardize column names
        col = col.lower()  # Ensure column name is in lowercase
        file_name = Path(file_path).stem  # Extract file name without extension

        # Check if required columns exist in the DataFrame
        missing_columns = required_columns - set(df.columns)
        if missing_columns:
            print(
                f"Warning: Missing required columns {missing_columns} in file {file_name}. Skipping..."
            )
            continue

        # Get unique values for filtering
        meas_numbers = df["meas_number"].unique()
        conditions = df["condition"].unique()
        tasks = df["task"].unique()

        # Iterate over all unique combinations of the selected columns
        for meas, cond, task in product(meas_numbers, conditions, tasks):
            # Filter DataFrame based on the current combination
            filtered_df = df[
                (df["meas_number"] == meas)
                & (df["condition"] == cond)
                & (df["task"] == task)
            ]

            # Check if the required column exists after filtering
            if col not in filtered_df.columns:
                print(
                    f"Warning: Column '{col}' not found in filtered data for {file_name}. Skipping..."
                )
                continue

            # Filter out infinite values before plotting
            filtered_corr = (
                filtered_df[col].replace([float("inf"), float("-inf")], None).dropna()
            )

            if filtered_corr.empty:
                print(
                    f"No valid data for {col} in {file_name} (meas={meas}, cond={cond}, task={task})"
                )
                continue

            # Create a percentage-based histogram for the specified column
            plt.figure(figsize=(8, 6))
            plt.hist(filtered_corr, bins=20, edgecolor="black", density=True)
            plt.title(f"{col} in {file_name} ({meas}{cond}{task})")
            plt.xlabel(col)
            plt.ylabel("Percentage")
            plt.grid(True)

            plots_dir = Path(plots_dir)
            result_plots_dir = plots_dir / file_name
            # Ensure the output directory exists
            result_plots_dir.mkdir(parents=True, exist_ok=True)

            # Save the plot
            plot_filename = f"{meas}{cond}{task}_{col}.png"
            plt.savefig(result_plots_dir / plot_filename)
            plt.close()

            print(f"Saved plot: {plot_filename} in {plots_dir}")

In [60]:
generate_histograms("corr", regular_files, hist_plots_dir / "corr")

Saved plot: 1Cbaseline1_corr.png in C:\Users\huber\OneDrive\Dokumenty\GitHub\swps_synchronization_study\results\plots\hist\corr
Saved plot: 1Cz1_1_f_corr.png in C:\Users\huber\OneDrive\Dokumenty\GitHub\swps_synchronization_study\results\plots\hist\corr
Saved plot: 1Cz1_2_m_corr.png in C:\Users\huber\OneDrive\Dokumenty\GitHub\swps_synchronization_study\results\plots\hist\corr
Saved plot: 1Cz1_3_f_corr.png in C:\Users\huber\OneDrive\Dokumenty\GitHub\swps_synchronization_study\results\plots\hist\corr
Saved plot: 1Cz1_4_m_corr.png in C:\Users\huber\OneDrive\Dokumenty\GitHub\swps_synchronization_study\results\plots\hist\corr
Saved plot: 1Cz1_5_f_corr.png in C:\Users\huber\OneDrive\Dokumenty\GitHub\swps_synchronization_study\results\plots\hist\corr
Saved plot: 1Cz1_6_m_corr.png in C:\Users\huber\OneDrive\Dokumenty\GitHub\swps_synchronization_study\results\plots\hist\corr
Saved plot: 1Cz1_corr.png in C:\Users\huber\OneDrive\Dokumenty\GitHub\swps_synchronization_study\results\plots\hist\corr
S